# Experiment 3 — Video Object Segmentation via Label Propagation (T05 refactor, T07 clean-up)

**Protocol:** non-parametric label propagation (Jabri et al. 2020, DINO context design), no training, frame-0 GT given. Video Swin-T per frame (zero-shot for AnyUp3D) → upsampler → 56² features → `davis.propagate_labels` → official DAVIS-2017 semi-supervised J&F (`metrics.eval_clip_jf`, first and last frame not scored, per-object aggregation).
**Conditions:** `bilinear`, `anyup2d` (per frame), `anyup3d`.

Built on the shared modules; see `docs/plan/P1_experiments/T05_exp2_exp3_refactor.md`.

- Protocol: 30 val clips, all frames, Swin-T Kinetics normalization, chunks of 16 (no overlap: the backbone is per frame), `PROP_SIZE=56`.
- The T05 regression check against the old Exp 3 code (`LEGACY`, passed) was removed in T07; it is in git at commit `8d34f11`.

No feature cache. Masks are written locally as palette PNGs (official results layout) and zipped to Drive; per-clip CSV + summary JSON go to `<OUTPUT_ROOT>/<RUN_NAME>/`. Seeding is deterministic (`anyup.utils.seed`). T4 is enough. **Runtime → Restart session** before running if `anyup` was imported before.

In [ ]:
# ── Cell 1 · Repo + Drive + deps ────────────────────────────────────────────
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS (T07); must be set before anything starts CUDA
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-in-action-test"   # T10 visual checks
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
N_CLIPS      = None    # None = all 30 val clips; int = first N (smoke test). In RUN_NAME, so a smoke run never overwrites a full one
CKPT_ANYUP3D = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # step6000 vs 6500 still open (plan.md)
OUTPUT_ROOT  = "/content/drive/MyDrive/results/T05/exp3"
MASK_LOCAL   = "/content/t05_exp3_masks"   # PNGs written here (fast), zipped to Drive at the end
DAVIS_DEST   = "/content"
DEVICE       = "cuda"
SEED         = 0

CONDITIONS   = ["bilinear", "anyup2d", "anyup3d"]
BASELINES    = ["bilinear", "anyup2d"]   # paired_stats: anyup3d vs each of these (if run)
SWIN_NORM    = "kinetics"                # Swin-T weights' own input stats (T01 default)

PROP_SIZE    = 56      # ↓ guide = upsampler output = propagation grid. Affinity per frame = PROP_SIZE² × (1+N_LAST)·PROP_SIZE² floats
                       #   (56: ≈ 275 MB). Dependent: N_LAST, Q_CHUNK, GPU memory of the clip features (N·768·PROP_SIZE²·4 B, ≤ 1 GB at 56)
CHUNK_SIZE   = 16      # ↓ frames per backbone / AnyUp3D call → peak memory and AnyUp3D temporal context. Dependent: Q_CHUNK, PROP_SIZE
OVERLAP      = 0       # Swin-T per frame: backbone-only overlap would not change any feature (Q9b). Must be < CHUNK_SIZE
Q_CHUNK      = None    # ↓ AnyUp3D query chunk (None = no chunking, as old Exp 3 at 56²). Dependent: CHUNK_SIZE, PROP_SIZE
TAU, TOP_K   = 0.07, 7 # propagation temperature / neighbours (Jabri et al.; DINO)
N_LAST       = 6       # ↓ context frames besides frame 0 → affinity memory. Dependent: PROP_SIZE

RUN_VISUALS       = False # optional stills + videos (Cell 6, T10). Standalone: run Cells 1–3, then 6 (reads the saved CSV + masks.zip)
VIZ_RUN_DIR       = None  # None = OUT_DIR; else a saved run folder, e.g. f"{OUTPUT_ROOT}/new_p56_c16_o0_qNone_kinetics" (T05 run)
VIZ_ITEMS         = None  # None = auto (largest AnyUp3D − AnyUp2D J&F gains + the worst clip); or a list of clip names
N_VISUAL_CLIPS    = 4     # ↓ rows of the stills figure = number of videos (masks are read from disk: cheap)
VISUAL_FRAME_FRAC = 0.8   # stills show the frame at this fraction of the clip (propagation errors grow over time)
VISUAL_FPS        = 8     # video speed (display only)
VISUAL_SIZE       = 240   # ↓ panel height in px (width keeps the aspect ratio) → video frame size. Independent of PROP_SIZE

print(f"CONDITIONS={CONDITIONS} N_CLIPS={N_CLIPS}")

In [ ]:
# ── Cell 3 · Imports, data, models ─────────────────────────────────────────
import random, time, shutil
from pathlib import Path
import numpy as np
import cv2
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

from anyup import davis, metrics
from anyup.backbones import BackboneLoader, BackboneOutput
from anyup.video import TensorSource, iter_backbone_chunks
from anyup.upsamplers import UpsamplerLoader
from anyup.utils.seed import set_seed   # shared: seeds + deterministic cuDNN / algorithms (T07)

set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0))

ROOT  = davis.download_davis(DAVIS_DEST)
CLIPS = davis.list_clips(ROOT, "val")
if N_CLIPS is not None:
    CLIPS = CLIPS[:N_CLIPS]
print(f"{len(CLIPS)} clips: {CLIPS}")
RUN_NAME  = f"p{PROP_SIZE}_c{CHUNK_SIZE}_o{OVERLAP}_q{Q_CHUNK}_{SWIN_NORM}_n{len(CLIPS)}"   # clip count: a smoke run can't overwrite a full run
OUT_DIR   = f"{OUTPUT_ROOT}/{RUN_NAME}"
MASK_ROOT = f"{MASK_LOCAL}/{RUN_NAME}"     # <MASK_ROOT>/<condition>/<clip>/<frame stem>.png
print("RUN_NAME:", RUN_NAME)

bb  = BackboneLoader.load("swin_t", device=DEVICE, guide_size=PROP_SIZE, temporal_mode="per_frame", norm=SWIN_NORM)
UPS = {}
if "bilinear" in CONDITIONS:
    UPS["bilinear"] = UpsamplerLoader.load("bilinear", device=DEVICE)
if "anyup2d" in CONDITIONS:
    UPS["anyup2d"] = UpsamplerLoader.load("anyup2d", device=DEVICE)
if "anyup3d" in CONDITIONS:
    UPS["anyup3d"] = UpsamplerLoader.load("anyup3d", ckpt_path=CKPT_ANYUP3D, device=DEVICE, q_chunk_size=Q_CHUNK)
print("upsamplers:", list(UPS))
shutil.rmtree(MASK_ROOT, ignore_errors=True)   # this run's masks only (no stale files)

In [ ]:
# ── Cell 4 · Main loop ──────────────────────────────────────────────────────
PER_CLIP = {c: {} for c in CONDITIONS}   # condition → clip → eval_clip_jf result
rows, frame_checks = [], []
t_start = time.perf_counter()
for ci, clip in enumerate(CLIPS):
    src_full = davis.open_clip(ROOT, clip)
    fp, ap   = davis.frame_paths(ROOT, clip), davis.ann_paths(ROOT, clip)
    idx      = list(range(len(src_full)))                                 # all frames
    src      = src_full                                                    # (T, 3, 480, W) [0, 1]
    gt       = davis.load_annotations(ROOT, clip)[idx]                     # (T, 480, W) uint8, native
    ann0     = davis.load_annotation(ap[idx[0]])                           # frame-0 GT, native

    chunks = list(iter_backbone_chunks(bb, src, chunk_size=CHUNK_SIZE, overlap=OVERLAP))   # Swin-T once per clip
    fidx = [f for ch in chunks for f in ch.meta["frame_index"]]
    frame_checks.append(fidx == list(range(len(idx))))

    for cond in CONDITIONS:
        t0 = time.perf_counter()
        feats = torch.cat([UPS[cond](ch, output_size=PROP_SIZE).features for ch in chunks])   # (T, 768, P, P) GPU
        pred  = davis.propagate_labels(feats, ann0, tau=TAU, top_k=TOP_K, n_last=N_LAST)      # (T, P, P) CPU
        mdir = Path(MASK_ROOT, cond, clip)
        for t, i in enumerate(idx):
            davis.save_mask_png(pred[t], ap[0], mdir / f"{fp[i].stem}.png")                  # native res + palette
        pred_native = np.stack([np.array(Image.open(mdir / f"{fp[i].stem}.png")) for i in idx])   # what the toolkit reads
        res = metrics.eval_clip_jf(gt, pred_native)
        PER_CLIP[cond][clip] = res
        j, f = float(np.mean(res["J_M"])), float(np.mean(res["F_M"]))
        rows.append(dict(clip=clip, condition=cond, J=j, F=f, JF=(j + f) / 2, n_objects=res["n_objects"],
                         n_frames=len(idx), n_scored=res["n_scored"], n_chunks=len(chunks),
                         time_s=round(time.perf_counter() - t0, 2)))
        print(f"[{ci + 1}/{len(CLIPS)}] {clip:<20} {cond:<9} J={j:.4f} F={f:.4f} J&F={(j + f) / 2:.4f} "
              f"objects={res['n_objects']} frames={len(idx)} chunks={len(chunks)} {time.perf_counter() - t0:.1f}s")
        del feats, pred
    del chunks
    torch.cuda.empty_cache()
print(f"done in {time.perf_counter() - t_start:.0f}s | frame_index == range(T) for every clip: {all(frame_checks)}")

In [ ]:
# ── Cell 5 · Summary, paired statistics, save ──────────────────────────────
summary = {"per_condition": {}, "paired_anyup3d_vs": {}, "frame_index_ok": all(frame_checks)}
keys = ["J&F-Mean", "J-Mean", "J-Recall", "J-Decay", "F-Mean", "F-Recall", "F-Decay"]
print(f"{'condition':<10} " + " ".join(f"{k:>9}" for k in keys))
for cond in CONDITIONS:
    s = metrics.summarize_jf(PER_CLIP[cond])
    summary["per_condition"][cond] = s
    print(f"{cond:<10} " + " ".join(f"{s[k]:>9.4f}" for k in keys) + f"   objects={s['n_objects']}")

fp_ = lambda p: "—" if p is None else f"{p:.2e}"
if "anyup3d" in CONDITIONS:
    ours = summary["per_condition"]["anyup3d"]["per_clip"]
    for base in BASELINES:
        if base not in CONDITIONS:
            continue
        b = summary["per_condition"][base]["per_clip"]
        st = metrics.paired_stats([ours[c]["JF"] for c in CLIPS], [b[c]["JF"] for c in CLIPS])
        summary["paired_anyup3d_vs"][base] = st
        print(f"anyup3d vs {base:<9}: ΔJ&F={st['mean_delta']:+.4f} improved {st['n_improved']}/{st['n']} "
              f"sign p2={fp_(st['sign_p_two_sided'])} wilcoxon p2={fp_(st['wilcoxon_p_two_sided'])}")

config = dict(N_CLIPS=N_CLIPS, clips=CLIPS, CKPT_ANYUP3D=CKPT_ANYUP3D, SEED=SEED,
              CONDITIONS=CONDITIONS, backbone="swin_t", temporal_mode="per_frame", SWIN_NORM=SWIN_NORM,
              PROP_SIZE=PROP_SIZE, CHUNK_SIZE=CHUNK_SIZE, OVERLAP=OVERLAP, Q_CHUNK=Q_CHUNK, TAU=TAU, TOP_K=TOP_K,
              N_LAST=N_LAST, RUN_NAME=RUN_NAME,
              anyup3d_meta=UPS["anyup3d"]._extra_meta() if "anyup3d" in UPS else None,
              anyup2d_meta=UPS["anyup2d"]._extra_meta() if "anyup2d" in UPS else None,
              repo_branch=REPO_BRANCH)
csv_path, json_path = metrics.save_results(OUT_DIR, rows, summary, config, name="exp3")
zip_path = shutil.make_archive(f"{OUT_DIR}/masks", "zip", MASK_ROOT)
print("saved:", csv_path, json_path, zip_path)

In [ ]:
# ── Cell 6 · Optional: stills + comparison videos (T10) ────────────────────
# Standalone: run Cells 1–3, then this cell. It reads per-clip J&F from Cell 4 (if run) or the saved CSV,
# and the masks from local disk (if Cell 4 ran) or <run>/masks.zip.
if RUN_VISUALS:
    import csv, json, zipfile
    from anyup import viz
    subprocess.run(["pip", "install", "imageio-ffmpeg", "-q"], check=True)   # H.264 writer for viz.write_mp4

    run_dir = Path(VIZ_RUN_DIR or OUT_DIR)
    vis_dir = run_dir / "visuals"
    vis_dir.mkdir(parents=True, exist_ok=True)
    conds = [c for c in ("bilinear", "anyup2d", "anyup3d") if c in CONDITIONS]
    base  = "anyup2d" if "anyup2d" in conds else conds[0]                  # strongest baseline = the gain reference

    # ── per-clip J&F (same numbers as the summary) ──
    if VIZ_RUN_DIR is None and globals().get("rows"):
        src_rows, source = rows, "in-memory rows (Cell 4)"
    else:
        with open(run_dir / "exp3_per_item.csv") as f:
            src_rows = list(csv.DictReader(f))
        source = str(run_dir / "exp3_per_item.csv")
    JF = {(r["condition"], r["clip"]): float(r["JF"]) for r in src_rows}
    avail = [c for c in dict.fromkeys(r["clip"] for r in src_rows) if all((k, c) in JF for k in conds)]
    print(f"J&F from {source}: {len(avail)} clips")

    # ── clip selection: largest AnyUp3D gains over `base` + the worst one (honest failure case) ──
    gain = lambda c: JF["anyup3d", c] - JF[base, c]
    if VIZ_ITEMS:
        sel, why = list(VIZ_ITEMS), {c: "manual (VIZ_ITEMS)" for c in VIZ_ITEMS}
    else:
        ranked = sorted(avail, key=gain, reverse=True)
        sel = ranked[:max(N_VISUAL_CLIPS - 1, 0)] + [ranked[-1]]
        why = {c: f"gain rank {i + 1}/{len(ranked)}" for i, c in enumerate(ranked)}
        why[ranked[-1]] = f"WORST gain (failure case), rank {len(ranked)}/{len(ranked)}"
    for c in sel:
        print(f"  {c:<20} " + " ".join(f"{k}={JF[k, c]:.3f}" for k in conds) + f"  Δ vs {base} {gain(c):+.3f}  [{why[c]}]")

    # ── masks: local (Cell 4 of this session) or unzip the selected clips from the saved run ──
    mask_root = Path(MASK_ROOT)
    if VIZ_RUN_DIR is not None or not all((mask_root / k / c).is_dir() for k in conds for c in sel):
        mask_root = Path(f"/content/t10_viz_masks/{run_dir.name}")
        with zipfile.ZipFile(run_dir / "masks.zip") as z:
            want = [m for m in z.namelist() if any(m.startswith(f"{k}/{c}/") for k in conds for c in sel)]
            z.extractall(mask_root, members=want)
        print(f"masks: {len(want)} PNGs unzipped from {run_dir / 'masks.zip'}")

    def frame_hw(clip):
        H0, W0 = np.array(Image.open(davis.frame_paths(ROOT, clip)[0])).shape[:2]
        return VISUAL_SIZE, int(round(W0 * VISUAL_SIZE / H0 / 2)) * 2      # keep aspect ratio, even width

    def load_rgb(path, hw):
        return viz.resize_rgb(np.array(Image.open(path).convert("RGB")), hw)

    def load_ids(path, hw):
        return davis.resize_labels(np.array(Image.open(path)), hw)            # nearest, ids kept

    def clip_panels(clip, i, hw):
        """dict: frame, gt, <cond> overlays of frame i (uint8 (h, w, 3))."""
        fp, ap = davis.frame_paths(ROOT, clip), davis.ann_paths(ROOT, clip)
        frame = load_rgb(fp[i], hw)
        out = {"frame": frame, "gt": viz.overlay(frame, load_ids(ap[i], hw))}
        for k in conds:
            out[k] = viz.overlay(frame, load_ids(mask_root / k / clip / f"{fp[i].stem}.png", hw))
        return out

    # ── stills: frame 0 with the given mask | GT at a late frame | each condition at that frame ──
    titles = ["frame 0: given mask", "ground truth"] + conds
    fig, axes = plt.subplots(len(sel), len(titles), figsize=(3.6 * len(titles), 2.4 * len(sel) + 0.6), squeeze=False)
    for r, clip in enumerate(sel):
        hw, n = frame_hw(clip), len(davis.frame_paths(ROOT, clip))
        t = min(n - 1, int(round(VISUAL_FRAME_FRAC * (n - 1))))
        p0, pt = clip_panels(clip, 0, hw), clip_panels(clip, t, hw)
        imgs = [p0["gt"], pt["gt"]] + [pt[k] for k in conds]
        caps = [f"{clip}\nframe 0: given mask", f"ground truth, frame {t}/{n - 1}"] + \
               [f"{k}  J&F {JF[k, clip]:.3f}" for k in conds]
        for c, (ax, img, cap) in enumerate(zip(axes[r], imgs, caps)):
            ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(cap, fontsize=10, fontweight="bold" if c == 2 + conds.index("anyup3d") else "normal")
        if "WORST" in why[clip]:
            for ax in axes[r]:
                for s in ax.spines.values():
                    s.set_edgecolor("red"); s.set_linewidth(2)
    fig.suptitle("Exp 3: object masks propagated from frame 0 using each upsampler's features "
                 "(red frame = worst AnyUp3D clip)", fontsize=12)
    plt.tight_layout()
    stills = vis_dir / "exp3_side_by_side.png"
    plt.savefig(stills, dpi=150, bbox_inches="tight"); plt.show()

    # ── one video per selected clip (MP4 for slides, GIF for anywhere) ──
    for clip in sel:
        hw, n = frame_hw(clip), len(davis.frame_paths(ROOT, clip))
        frames = []
        for i in range(n):
            p = clip_panels(clip, i, hw)
            row = [viz.label_strip(p["gt"], f"ground truth  frame {i}/{n - 1}")] + \
                  [viz.label_strip(p[k], f"{k}  J&F {JF[k, clip]:.3f}") for k in conds]
            frames.append(viz.label_strip(viz.grid([row]), f"{clip}: mask given on frame 0 only, "
                                          f"propagated with each upsampler's features", height=26))
        mp4 = viz.write_mp4(frames, vis_dir / f"exp3_{clip}.mp4", fps=VISUAL_FPS)
        gif = viz.write_gif(frames, vis_dir / f"exp3_{clip}.gif", fps=VISUAL_FPS)
        print(f"saved {mp4.name} / {gif.name}  ({n} frames)")

    with open(vis_dir / "visuals_selection.json", "w") as f:
        json.dump(dict(source=source, base=base, frame_frac=VISUAL_FRAME_FRAC,
                       clips=[dict(clip=c, why=why[c], **{k: JF[k, c] for k in conds}) for c in sel]), f, indent=2)
    print("saved:", stills, "| videos + visuals_selection.json in", vis_dir)
else:
    print("RUN_VISUALS=False — skipped")